# MyDicomViewer AI — モデルの評価

アプリに入れた **ONNX モデルそのもの** を、学習に使っていないテストデータで評価します。

準備（右パネル）:
- **Input** に RSNA Pneumonia Detection Challenge が入っていること
- **Upload** で `pneumonia.onnx` と `model_meta.json` を追加（データセット名は例えば `pneumonia-model`）
- **Internet = ON**（GPU は不要です）

上から順に実行してください。

In [ ]:
# 評価用コードを作業フォルダに書き出す
import os
os.chdir('/kaggle/working')

open('preprocess.py', 'w', encoding='utf-8').write(r'''"""DICOM → モデル入力 の前処理。

学習時・推論時(C# 側を含む)で完全に同じ処理をする必要があるため、
ここに一元化しておく。C# 側はこのファイルの手順をそのまま移植すること。

手順:
  1. pixel_array に RescaleSlope / RescaleIntercept を適用
  2. MONOCHROME1 の場合は白黒反転
  3. 画像全体の min-max で 0〜1 に正規化
  4. 224x224 に縮小(INTER_AREA)して 0〜255 の uint8 に
  5. (推論直前) /255 → 3ch に複製 → ImageNet の mean/std で標準化
"""
from __future__ import annotations

import cv2
import numpy as np
import pydicom

IMG_SIZE = 224
MEAN = (0.485, 0.456, 0.406)
STD = (0.229, 0.224, 0.225)


def dicom_to_uint8(path: str, size: int = IMG_SIZE) -> np.ndarray:
    """DICOM ファイルを読み、size x size の uint8 グレースケール画像を返す。"""
    ds = pydicom.dcmread(path)
    img = ds.pixel_array.astype(np.float32)
    if img.ndim != 2:
        raise ValueError(f"2次元のグレースケール画像のみ対応しています: shape={img.shape}")

    slope = float(getattr(ds, "RescaleSlope", 1.0))
    intercept = float(getattr(ds, "RescaleIntercept", 0.0))
    img = img * slope + intercept

    if getattr(ds, "PhotometricInterpretation", "") == "MONOCHROME1":
        img = img.max() - img

    lo, hi = float(img.min()), float(img.max())
    img = (img - lo) / (hi - lo) if hi > lo else np.zeros_like(img)

    img = cv2.resize(img, (size, size), interpolation=cv2.INTER_AREA)
    return np.clip(img * 255.0, 0, 255).round().astype(np.uint8)


def uint8_to_input(img: np.ndarray) -> np.ndarray:
    """uint8 (H, W) → モデル入力 float32 (1, 3, H, W)。"""
    x = img.astype(np.float32) / 255.0
    x = np.repeat(x[None, :, :], 3, axis=0)
    mean = np.array(MEAN, dtype=np.float32)[:, None, None]
    std = np.array(STD, dtype=np.float32)[:, None, None]
    return ((x - mean) / std)[None]
''')

open('evaluate.py', 'w', encoding='utf-8').write(r'''"""書き出した ONNX モデルを、学習に使っていないテストデータで評価する。

評価すること:
  1. 分類性能: AUC（ブートストラップ法による 95% 信頼区間つき）、感度・特異度・混同行列
     → ONNX 変換後も学習時（PyTorch）と同じ性能が出ているかの確認も兼ねる
  2. 位置の妥当性: Grad-CAM が、放射線科医が付けた病変の枠（RSNA のバウンディングボックス）を
     指しているか
       - Pointing Game: ヒートマップの最大点が病変の枠の中にある割合
       - Energy Ratio : ヒートマップの総量のうち、病変の枠の中にある割合
     どちらも「枠の面積の割合（ランダムに選んだ場合の期待値）」と比べる

テストデータの分け方は train.py と完全に同じ（患者単位・同じ乱数シード）。

使い方 (Kaggle):
    python evaluate.py --data-dir <コンペのデータ> --model-dir <pneumonia.onnx のあるフォルダ>
"""
from __future__ import annotations

import argparse
import json
import time
from pathlib import Path

import numpy as np
import onnxruntime as ort
import pandas as pd
import pydicom
from sklearn.metrics import confusion_matrix, roc_auc_score, roc_curve
from sklearn.model_selection import train_test_split

from preprocess import IMG_SIZE, dicom_to_uint8, uint8_to_input


def test_split(data_dir: Path, seed: int) -> tuple[pd.DataFrame, pd.DataFrame]:
    """train.py と同じ手順でテスト用の患者を選ぶ。戻り値: (テスト患者, 病変の枠)"""
    raw = pd.read_csv(data_dir / "stage_2_train_labels.csv")
    df = raw.groupby("patientId", as_index=False)["Target"].max().sort_values("patientId").reset_index(drop=True)
    idx = np.arange(len(df))
    y = df["Target"].to_numpy()
    _, tmp_idx = train_test_split(idx, test_size=0.2, stratify=y, random_state=seed)
    _, te_idx = train_test_split(tmp_idx, test_size=0.5, stratify=y[tmp_idx], random_state=seed)
    test = df.iloc[te_idx].reset_index(drop=True)
    boxes = raw[(raw["Target"] == 1) & raw["patientId"].isin(test["patientId"])]
    return test, boxes


def predict(session: ort.InferenceSession, paths: list[str], batch_size: int) -> tuple[np.ndarray, np.ndarray]:
    probs, cams = [], []
    for start in range(0, len(paths), batch_size):
        batch = np.concatenate([uint8_to_input(dicom_to_uint8(p)) for p in paths[start:start + batch_size]])
        prob, cam = session.run(["prob", "cam"], {"input": batch})
        probs.append(prob[:, 0])
        cams.append(cam[:, 0])
        if (start // batch_size) % 10 == 0:
            print(f"  {min(start + batch_size, len(paths))} / {len(paths)}")
    return np.concatenate(probs), np.concatenate(cams)


def bootstrap_auc(labels: np.ndarray, probs: np.ndarray, n: int, seed: int) -> tuple[float, float]:
    rng = np.random.default_rng(seed)
    scores = []
    for _ in range(n):
        i = rng.integers(0, len(labels), len(labels))
        if labels[i].min() != labels[i].max():  # 片方のクラスしかない標本は AUC が定義できない
            scores.append(roc_auc_score(labels[i], probs[i]))
    return float(np.percentile(scores, 2.5)), float(np.percentile(scores, 97.5))


def localization(cam: np.ndarray, patient_boxes: pd.DataFrame, width: int, height: int) -> dict:
    """1枚分の位置評価。枠は元画像の画素座標なので、CAM（224x224）の座標に縮める。"""
    mask = np.zeros_like(cam, dtype=bool)
    sx, sy = cam.shape[1] / width, cam.shape[0] / height
    for _, b in patient_boxes.iterrows():
        x0, y0 = int(np.floor(b.x * sx)), int(np.floor(b.y * sy))
        x1, y1 = int(np.ceil((b.x + b.width) * sx)), int(np.ceil((b.y + b.height) * sy))
        mask[max(y0, 0):y1, max(x0, 0):x1] = True

    peak_y, peak_x = np.unravel_index(np.argmax(cam), cam.shape)
    total = float(cam.sum())
    return {
        "hit": bool(mask[peak_y, peak_x]),
        "energy": float(cam[mask].sum() / total) if total > 0 else 0.0,
        "area": float(mask.mean()),
    }


def main() -> None:
    p = argparse.ArgumentParser()
    p.add_argument("--data-dir", type=Path, required=True)
    p.add_argument("--model-dir", type=Path, required=True)
    p.add_argument("--out-dir", type=Path, default=Path("outputs"))
    p.add_argument("--seed", type=int, default=42)
    p.add_argument("--batch-size", type=int, default=32)
    p.add_argument("--bootstrap", type=int, default=1000)
    args = p.parse_args()

    meta = json.loads((args.model_dir / "model_meta.json").read_text(encoding="utf-8"))
    threshold = float(meta["threshold"])
    session = ort.InferenceSession(str(args.model_dir / "pneumonia.onnx"), providers=["CPUExecutionProvider"])

    test, boxes = test_split(args.data_dir, args.seed)
    img_dir = args.data_dir / "stage_2_train_images"
    paths = [str(img_dir / f"{pid}.dcm") for pid in test["patientId"]]
    labels = test["Target"].to_numpy()
    print(f"テスト: {len(test)} 人（陽性 {labels.sum()} 人）。推論中...")

    t0 = time.time()
    probs, cams = predict(session, paths, args.batch_size)
    elapsed = time.time() - t0

    # ---- 1. 分類性能
    pred = probs >= threshold
    tn, fp, fn, tp = confusion_matrix(labels, pred, labels=[0, 1]).ravel()
    auc = float(roc_auc_score(labels, probs))
    ci_low, ci_high = bootstrap_auc(labels, probs, args.bootstrap, args.seed)
    classification = {
        "auc": auc, "auc_95ci": [ci_low, ci_high],
        "sensitivity": tp / (tp + fn), "specificity": tn / (tn + fp),
        "ppv": tp / (tp + fp) if tp + fp else None, "npv": tn / (tn + fn) if tn + fn else None,
        "confusion_matrix": {"tp": int(tp), "fp": int(fp), "tn": int(tn), "fn": int(fn)},
        "threshold": threshold, "n": int(len(labels)), "positives": int(labels.sum()),
        "seconds_per_image_cpu": elapsed / len(labels),
    }

    # ---- 2. 位置の妥当性（陽性かつ枠のある画像のみ）
    results = []
    for i, pid in enumerate(test["patientId"]):
        if labels[i] != 1:
            continue
        patient_boxes = boxes[boxes["patientId"] == pid]
        if patient_boxes.empty:
            continue
        header = pydicom.dcmread(paths[i], stop_before_pixels=True)
        results.append(localization(cams[i], patient_boxes, int(header.Columns), int(header.Rows)))

    hits = np.array([r["hit"] for r in results])
    energy = np.array([r["energy"] for r in results])
    area = np.array([r["area"] for r in results])
    localization_summary = {
        "n": len(results),
        "pointing_game": float(hits.mean()),
        "energy_ratio": float(energy.mean()),
        "random_baseline_box_area": float(area.mean()),
    }

    # ---- 保存
    args.out_dir.mkdir(parents=True, exist_ok=True)
    report = {"model": meta.get("model", "unknown"), "classification": classification, "localization": localization_summary}
    (args.out_dir / "evaluation.json").write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8")

    fpr, tpr, _ = roc_curve(labels, probs)
    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
        fig, ax = plt.subplots(figsize=(5, 5), dpi=150)
        ax.plot(fpr, tpr, color="#7B4FA6", lw=2, label=f"AUC {auc:.3f} (95% CI {ci_low:.3f}–{ci_high:.3f})")
        ax.plot([0, 1], [0, 1], color="#999999", lw=1, ls="--")
        ax.scatter([1 - classification["specificity"]], [classification["sensitivity"]], color="#D9534F", zorder=3,
                   label=f"Operating point (threshold {threshold:.3f})")
        ax.set_xlabel("1 - Specificity")
        ax.set_ylabel("Sensitivity")
        ax.set_title("ROC curve (held-out test set)")
        ax.legend(loc="lower right", fontsize=8)
        fig.tight_layout()
        fig.savefig(args.out_dir / "roc_curve.png")
    except ImportError:
        print("matplotlib が無いため ROC 曲線の画像は省略しました")

    print(json.dumps(report, indent=2, ensure_ascii=False))


if __name__ == "__main__":
    main()
''')

print('書き出し完了:', sorted(f for f in os.listdir('.') if f.endswith('.py')))


## 1. データとモデルの場所を確認

In [ ]:
import glob, os
labels = glob.glob('/kaggle/input/**/stage_2_train_labels.csv', recursive=True)
models = glob.glob('/kaggle/input/**/pneumonia.onnx', recursive=True)
assert labels, 'コンペのデータが見つかりません。Add Input で RSNA Pneumonia Detection Challenge を追加してください。'
assert models, 'pneumonia.onnx が見つかりません。右パネルの Upload で pneumonia.onnx と model_meta.json を追加してください。'
DATA_DIR = os.path.dirname(labels[0])
MODEL_DIR = os.path.dirname(models[0])
assert os.path.exists(os.path.join(MODEL_DIR, 'model_meta.json')), 'model_meta.json が pneumonia.onnx と同じ場所にありません。'
print('DATA_DIR  =', DATA_DIR)
print('MODEL_DIR =', MODEL_DIR)

## 2. 評価を実行
テスト約2,700枚を CPU で推論します（数分かかります）。

In [ ]:
!pip install -q onnxruntime
!python evaluate.py --data-dir "{DATA_DIR}" --model-dir "{MODEL_DIR}"

## 3. 結果

In [ ]:
import json
from IPython.display import Image, display

r = json.load(open('outputs/evaluation.json', encoding='utf-8'))
c, l = r['classification'], r['localization']
print(f"AUC            {c['auc']:.3f}  (95% CI {c['auc_95ci'][0]:.3f}–{c['auc_95ci'][1]:.3f})")
print(f"感度 / 特異度   {c['sensitivity']:.3f} / {c['specificity']:.3f}")
print(f"陽性的中率      {c['ppv']:.3f}   陰性的中率 {c['npv']:.3f}")
print(f"推論時間(CPU)   {c['seconds_per_image_cpu']*1000:.0f} ms / 枚")
print()
print(f"位置の評価（陽性 {l['n']} 枚）")
print(f"  Pointing Game  {l['pointing_game']:.3f}  (ランダムなら {l['random_baseline_box_area']:.3f})")
print(f"  Energy Ratio   {l['energy_ratio']:.3f}  (ランダムなら {l['random_baseline_box_area']:.3f})")
display(Image('outputs/roc_curve.png', width=420))

## 4. ダウンロード
右パネルの **Output** → `outputs` から次の2つをダウンロードしてください。
- `evaluation.json`
- `roc_curve.png`